# Algorithm Analysis: PN Solution of the Milne Problem

This notebook walks through the `milne_pn` package interactively: building the PN
system, solving for the extrapolation distance $z_0$, and -- the main point of this
notebook -- connecting the **slow convergence of $z_0$** to the **Runge/Gibbs
phenomenon** in the truncated angular-flux reconstruction.

A desktop GUI covering the same ground interactively (with live plots and a sortable
data table) is available via `python -m milne_pn.gui` -- see `docs/architecture.md`.

See `docs/algorithms.md` for the full derivation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from milne_pn.algorithms.pn_system import PNSystem
from milne_pn.algorithms.milne_solver import MilneSolver
from milne_pn.algorithms.runge_analysis import (
    reconstruct_boundary_flux, gibbs_overshoot, runge_interpolation_demo
)

%matplotlib inline

## 1. The PN matrices for a single order

In [ ]:
N = 5
pn = PNSystem(N)
print("A (tridiagonal, zero diagonal):")
print(np.round(pn.A, 3))
print("\nD (diag(0,1,1,...,1)):")
print(np.round(pn.D, 3))

## 2. Solve and read off z0 directly (no fitting)

In [ ]:
solver = MilneSolver(21)
print(f"N = {solver.N}")
print(f"z0 = {solver.z0:.6f}  (Case exact: 0.710446)")
print(f"discrete decaying modes kept: {solver.num_discrete_modes}")
print(f"mode eigenvalues (omega_i): {np.round(solver.omega, 4)}")

## 3. Convergence of $z_0$ with PN order

Slow, roughly $O(1/N)$: dozens of orders are needed for 3-4 correct digits.

In [ ]:
from milne_pn.benchmarks.benchmark import convergence_benchmark

result = convergence_benchmark(n_max=79)
fig, ax = plt.subplots(figsize=(6,4))
ax.axhline(0.710446, color="crimson", ls="--", label="Case exact")
ax.plot(result["N"], result["z0"], "o-", ms=4)
ax.set_xlabel("PN order N"); ax.set_ylabel("z0"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 4. Why so slow? The Runge/Gibbs phenomenon

The *exact* boundary angular flux $\psi(0,\mu)$ has a jump discontinuity at
$\mu=0$: zero for $\mu>0$ (vacuum), finite for $\mu<0$ (outgoing). The PN method
reconstructs $\psi(0,\mu)$ as a *finite Legendre series* -- a global polynomial
approximation to a discontinuous function. That is exactly the Gibbs-phenomenon
setting (the angular-domain sibling of the classical Runge phenomenon): the
reconstruction **rings** near $\mu=0$, and the ring amplitude does not shrink
with N -- only its width does.

In [ ]:
orders = [3, 9, 21, 41]
fig, ax = plt.subplots(figsize=(7,4.5))
for Nord in orders:
    s = MilneSolver(Nord)
    mu, psi0 = reconstruct_boundary_flux(s, n_mu=2000)
    ax.plot(mu, psi0, lw=1.3, label=f"P{Nord}")
ax.axvline(0, color="k", lw=0.8)
ax.axvspan(0, 1, color="red", alpha=0.05)
ax.set_xlabel("mu"); ax.set_ylabel("psi(0,mu)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Gibbs ringing at the angular discontinuity")
plt.show()

for Nord in orders:
    stats = gibbs_overshoot(MilneSolver(Nord))
    print(f"N={Nord:3d}  relative overshoot = {stats['relative_overshoot']*100:5.1f}%")

Notice the relative overshoot **plateaus** rather than decaying to zero --
that plateau is the Gibbs signature, and it's the same mechanism (applied to a
different basis/domain) as the classical Runge phenomenon below.

In [ ]:
result = runge_interpolation_demo(degrees=(5,10,15,20))
x = result["x_eval"]
fig, ax = plt.subplots(figsize=(7,4.5))
ax.plot(x, result["y_true"], "k-", lw=2, label="f(x) = 1/(1+25x^2)")
for d in (5,10,15,20):
    ax.plot(x, result["interpolants"][d], lw=1.2, label=f"degree {d}")
ax.set_ylim(-1,2); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title("Classical Runge phenomenon (equispaced-node interpolation)")
plt.show()

## 5. Takeaway

Both plots show the same underlying pathology: a finite-order **global** polynomial representation of a non-smooth (discontinuous or steeply peaked) function rings/oscillates, with an overshoot that does not vanish as the order increases. This is why increasing PN order alone buys only algebraic (not exponential) convergence for $z_0$ -- confirmed quantitatively above.

To explore this interactively -- change N with a slider-like spinbox, rerun the convergence sweep, and browse the results in a sortable table -- try the desktop GUI: `python -m milne_pn.gui`.